# Hyperparameter search

[General optimization](general-optimization.ipynb) tuned a model's *parameters*.
**Hyperparameters** are the knobs you set *before* training — regularization
strength, tree depth, *k* in k-means. You can't learn them by gradient descent
on the training loss; you search for the values that give the best
**validation** score.

```{note}
Below, `val_error(hp)` is a stand-in for *"train a model with this
hyperparameter and measure its cross-validated error"* (from the [Model
Evaluation](../01d-evaluation/cross-validation.ipynb) chapter). We use a cheap
synthetic objective so the *search strategies* — not model fitting — are the
focus. Its minimum sits near `hp = 2.0`.
```

In [ ]:
:dep rand = { version = "0.10" }

// Lower is better. In practice this would fit a model and return CV error.
fn val_error(hp: f64) -> f64 {
    (hp - 2.0).powi(2) * 0.1 + 0.05 + 0.02 * (hp * 3.0).sin()
}
println!("objective ready (search the range [0, 5]; lower is better)");

## 1. Grid search

The simplest strategy: evaluate every point on a regular grid. Exhaustive and
reproducible, but the cost explodes with the number of hyperparameters (the
"curse of dimensionality").

In [ ]:
{
    let (lo, hi, n) = (0.0_f64, 5.0, 11usize);
    let mut best = (f64::INFINITY, 0.0_f64);
    for i in 0..n {
        let hp = lo + (hi - lo) * i as f64 / (n - 1) as f64;
        let e = val_error(hp);
        if e < best.0 { best = (e, hp); }
    }
    println!("grid search  : best hp = {:.2}, error = {:.4}  ({} evals)", best.1, best.0, 11);
}

## 2. Random search

Sample the space at random instead of on a grid. Counter-intuitively this
usually **beats** grid search in higher dimensions: with a fixed budget it tries
more distinct values of each individual hyperparameter, rather than wasting
evaluations on redundant grid combinations.

In [ ]:
// In rand 0.10 the sampling helpers live on the `RngExt` trait.
use rand::{RngExt, SeedableRng};

{
    let mut rng = rand::rngs::StdRng::seed_from_u64(0);
    let mut best = (f64::INFINITY, 0.0_f64);
    for _ in 0..11 {
        let hp: f64 = rng.random_range(0.0..5.0);
        let e = val_error(hp);
        if e < best.0 { best = (e, hp); }
    }
    println!("random search: best hp = {:.2}, error = {:.4}  (11 evals)", best.1, best.0);
}

## 3. Bayesian search with TPE

A **Tree-structured Parzen Estimator** ([`tpe`](https://docs.rs/tpe)) is smarter:
it models which regions of the space have produced good scores so far and
concentrates new trials there. With the same tiny budget it should home in on
the optimum more reliably. The `ask`/`tell` loop drives it:

In [ ]:
:dep tpe = { version = "0.3" }
use tpe::{TpeOptimizer, parzen_estimator, range};

{
    let mut optim = TpeOptimizer::new(parzen_estimator(), range::Range::new(0.0, 5.0).unwrap());
    let mut rng = rand::rngs::StdRng::seed_from_u64(0);
    let mut best = (f64::INFINITY, 0.0_f64);
    for _ in 0..11 {
        let hp = optim.ask(&mut rng).unwrap();
        let e = val_error(hp);
        optim.tell(hp, e).unwrap();
        if e < best.0 { best = (e, hp); }
    }
    println!("TPE search   : best hp = {:.2}, error = {:.4}  (11 evals)", best.1, best.0);
}

All three land near `hp = 2.0`; on a harder, higher-dimensional objective the
gap between them widens in favour of random and TPE.

```{warning}
**Ecosystem maturity.** Rust's hyperparameter-optimization tooling is markedly
thinner than Python's — there is no direct equivalent of Optuna / Hyperopt /
Ray Tune with the same breadth. `tpe` is a focused, single-algorithm crate, not
a full HPO framework, and other crates in this space are early-stage. Check the
current state on crates.io before relying on version-specific capabilities.
```

Grid and random search are just a few lines by hand, which is often all you
need. Next: [AutoML](../06-automl/automl-classification.ipynb) — which automates
the search over *model choice*, complementary to this chapter's search over one
model's hyperparameters.